In [10]:
import numpy as np
from scipy.spatial import ConvexHull
import json, math


In [11]:
def perimeter_convex(pts):
    if len(pts) < 3:
        return np.nan
    try:
        hull = ConvexHull(pts)
    except Exception:
        return np.nan
    return float(sum(np.linalg.norm(pts[hull.vertices[i]] - pts[hull.vertices[(i+1)%len(hull.vertices)]])
                     for i in range(len(hull.vertices))))

def _slice_points_xy(vertices, y0, tol=0.008):
    y = vertices[:, 1]
    m = np.abs(y - y0) < tol
    pts = vertices[m][:, [0, 2]]
    if pts.shape[0] < 3:
        return np.empty((0, 2))
    return pts - pts.mean(axis=0)

def slice_points(vertices, y0, tol=0.0075):
    y = vertices[:, 1]
    m = np.abs(y - y0) < tol
    pts = vertices[m][:, [0, 2]]
    if pts.shape[0] < 3:
        return np.empty((0, 2))
    return pts - pts.mean(axis=0)

def scan_perimeter_between(vertices, y_min, y_max, step=0.005):
    if y_max <= y_min:
        return np.empty((0, 2))
    ys = np.arange(y_min, y_max, step)
    if ys.size == 0:
        ys = np.linspace(y_min, y_max, 3)
    vals = []
    for y0 in ys:
        per = perimeter_convex(slice_points(vertices, y0))
        vals.append([y0, per])
    return np.asarray(vals, float)


In [12]:
def orth_slice_along_segment_robust(vertices, p0, p1,
                                    t_list=(0.3, 0.4, 0.5, 0.6),
                                    radius=0.14, tol=0.012,
                                    horiz_fallback=True):
    v = p1 - p0
    nv = np.linalg.norm(v)
    if nv < 1e-6:
        return np.nan
    v = v / nv

    a = np.array([1.0, 0.0, 0.0])
    if abs(a @ v) > 0.9:
        a = np.array([0.0, 1.0, 0.0])
    u = np.cross(v, a); u = u / np.linalg.norm(u)
    w = np.cross(v, u)

    best = np.nan
    for t in t_list:
        c = p0 + t * (p1 - p0)
        X = vertices - c
        proj = X @ v
        ortho = X - np.outer(proj, v)
        dist = np.linalg.norm(ortho, axis=1)

        tube = dist < radius
        if not np.any(tube):
            continue
        slab = np.abs(proj[tube]) < tol
        pts3 = (vertices[tube])[slab]
        if pts3.shape[0] < 3:
            continue

        rel = pts3 - c
        pts2 = np.stack([rel @ u, rel @ w], axis=1)
        per = perimeter_convex(pts2)
        if np.isfinite(per):
            best = per if not np.isfinite(best) else max(best, per)

    if not np.isfinite(best) and horiz_fallback:
        y_mid = 0.5 * (p0[1] + p1[1])
        pts2 = _slice_points_xy(vertices, y_mid, tol=max(0.010, tol))
        per = perimeter_convex(pts2)
        if np.isfinite(per):
            best = per
    return best


In [13]:
def body_height_y(vertices):
    return float(vertices[:,1].max()-vertices[:,1].min())

def rescale_to_user_height(vertices, joints, user_height_m, width_scale=1.0):
    h = body_height_y(vertices)
    s = user_height_m / max(h, 1e-6)
    V = vertices * s
    J = joints   * s
    if abs(width_scale - 1.0) > 1e-6:
        V = V.copy()
        V[:, [0, 2]] *= float(width_scale)
    return V, J, s

def avg_y_mp(joints, mp, a, b):
    return 0.5*(joints[mp[a],1]+joints[mp[b],1])

def ground_y_from_ankles_or_mesh(joints, vertices, mp):
    ankles = float(min(joints[mp["left_ankle"],1], joints[mp["right_ankle"],1]))
    floor  = float(vertices[:,1].min())
    return min(ankles, floor)

def ground_y_from_ankles_only(joints, mp):
    return float(min(joints[mp["left_ankle"],1], joints[mp["right_ankle"],1]))


def choose_joint_map_fixed_custom():
    return {
        "pelvis": 9,
        "left_hip": 8,  "right_hip": 12,
        "left_knee": 3, "right_knee": 4,
        "left_ankle": 7,"right_ankle": 6,
        "neck": 22,
        "left_shoulder": 20, "right_shoulder": 21,
        "left_elbow": 18,    "right_elbow": 19,
        "left_wrist": 16,    "right_wrist": 17,
    }


In [14]:
def measure_core(vertices, joints):
    res = {}
    mp = choose_joint_map_fixed_custom()
    print(">>> Using CUSTOM joint map (pelvis=9, hips=8/12, knees=3/4, ankles=7/6, neck=22).")

    y_neck   = float(joints[mp["neck"],1])
    y_pelvis = float(joints[mp["pelvis"],1])
    y_mid    = 0.5*(y_neck + y_pelvis)

    # --- Chest
    scan_chest = scan_perimeter_between(vertices, y_mid, y_neck-0.02, step=0.004)
    res["chest_circum_cm"] = float(np.nanmean(scan_chest[:,1])*100) if scan_chest.size else np.nan

    # --- Waist
    scan_waist = scan_perimeter_between(vertices, y_pelvis+0.03, y_mid, step=0.004)
    res["waist_circum_cm"] = float(np.nanmean(scan_waist[:,1])*100) if scan_waist.size else np.nan

    # --- Hip
    y_hips_mean = avg_y_mp(joints, mp, "left_hip", "right_hip")
    scan_hip = scan_perimeter_between(vertices, y_hips_mean-0.18, y_hips_mean+0.05, step=0.004)
    res["hip_circum_cm"] = float(np.nanmean(scan_hip[:,1])*100) if scan_hip.size else np.nan

    # --- Arms
    L_sh, L_el = joints[mp["left_shoulder"]],  joints[mp["left_elbow"]]
    R_sh, R_el = joints[mp["right_shoulder"]], joints[mp["right_elbow"]]
    res["upperarm_circum_cm"] = float(np.nanmean([
        orth_slice_along_segment_robust(vertices,L_sh,L_el),
        orth_slice_along_segment_robust(vertices,R_sh,R_el)
    ])*100.0)

    # --- Thigh
    L_hp, L_kn = joints[mp["left_hip"]], joints[mp["left_knee"]]
    R_hp, R_kn = joints[mp["right_hip"]], joints[mp["right_knee"]]
    res["thigh_circum_cm"] = float(np.nanmean([
        orth_slice_along_segment_robust(vertices,L_hp,L_kn),
        orth_slice_along_segment_robust(vertices,R_hp,R_kn)
    ])*100.0)

    # --- Outseam
    y_hips_mean = avg_y_mp(joints, mp, "left_hip", "right_hip")
    y_ground = ground_y_from_ankles_or_mesh(joints, vertices, mp)
    res["pants_outseam_cm"] = float(max(0.0, y_hips_mean - y_ground)*100.0)

    # --- Sleeve / Top / Gown
    res["sleeve_len_cm"] = 70.0  # مقدار جایگزین تا تثبیت مفاصل
    res["top_len_cm"] = float(max(0.0, y_neck - (y_hips_mean+0.02))*100)
    res["gown_len_cm"] = float(max(0.0, y_neck - y_ground)*100)
    return res


In [15]:
def measure_pipeline(smpl_output_tpose, user_height_cm=175, gender="male", width_scale=0.75):
    vertices = smpl_output_tpose["vertices"]
    joints   = smpl_output_tpose["joints"]
    vertices, joints, s = rescale_to_user_height(vertices, joints, user_height_cm/100.0, width_scale)
    print("Scaled body height (m):", body_height_y(vertices))

    res = measure_core(vertices, joints)
    res.update({
        "gender": gender,
        "user_height_cm": user_height_cm,
        "scale_factor": s,
        "width_scale": width_scale,
    })
    return res


In [16]:
def print_measurement_results_farsi(results):
    labels = {
        "chest_circum_cm": "دور سینه",
        "waist_circum_cm": "دور کمر",
        "hip_circum_cm": "دور باسن",
        "upperarm_circum_cm": "دور بازو",
        "thigh_circum_cm": "دور ران",
        "pants_outseam_cm": "قد بیرونی شلوار",
        "sleeve_len_cm": "قد آستین",
        "top_len_cm": "قد بالاتنه (بلوز)",
        "gown_len_cm": "قد لباس بلند",
        "user_height_cm": "قد کاربر",
        "scale_factor": "ضریب مقیاس مدل",
        "width_scale": "ضریب پهنای بدن",
        "gender": "جنسیت",
    }
    print("📏 نتایج اندازه‌گیری بدن:\n" + "-"*40)
    for k, label in labels.items():
        if k in results:
            v = results[k]
            if isinstance(v, (int, float)) and np.isfinite(v):
                unit = "سانتی‌متر" if k.endswith("_cm") else ""
                print(f"{label}: {v:.1f} {unit}")
            elif isinstance(v, str):
                print(f"{label}: {v}")
    print("-"*40)


In [17]:
# # Auto width_scale from user's weight & height (BMI-based)
#
# def width_scale_from_weight_height(weight_kg: float,
#                                    height_cm: float,
#                                    *,
#                                    base_width: float = 0.75,
#                                    bmi_ref: float = 23.0,
#                                    power: float = 0.35,
#                                    min_scale: float = 0.60,
#                                    max_scale: float = 1.15) -> float:
#     """
#     محاسبه‌ی width_scale بر پایه‌ی BMI کاربر.
#     - base_width: مقدار مرجع برای فرد با BMI≈bmi_ref (ما قبلاً 0.75 استفاده می‌کردیم)
#     - power: شدت حساسیت نسبت به BMI (۰.۳۵ ملایم و پایدار است)
#     - min/max: کران‌های منطقی برای جلوگیری از افراط
#
#     مثال:
#       BMI 23  -> ~0.75
#       BMI 26  -> ~0.79
#       BMI 30  -> ~0.83
#       BMI 18  -> ~0.70 (باریک‌تر)
#     """
#     h_m = max(0.01, float(height_cm) / 100.0)
#     bmi = float(weight_kg) / (h_m ** 2)
#     ratio = (bmi / bmi_ref) ** power
#     ws = base_width * ratio
#     ws = max(min_scale, min(max_scale, ws))
#     return float(ws), float(bmi)

# تنظیمات ملایم‌تر برای بدن‌های لاغر
def width_scale_from_weight_height(weight_kg: float,
                                   height_cm: float,
                                   *,
                                   base_width: float = 0.80,   # قبلاً 0.75 بود
                                   bmi_ref: float = 23.0,
                                   power: float = 0.25,        # قبلاً 0.35 بود
                                   min_scale: float = 0.70,    # قبلاً 0.60
                                   max_scale: float = 1.15) -> float:
    h_m = max(0.01, float(height_cm)/100.0)
    bmi = float(weight_kg)/(h_m**2)
    ratio = (bmi/bmi_ref)**power
    ws = base_width*ratio
    ws = max(min_scale, min(max_scale, ws))
    return float(ws), float(bmi)



In [19]:
# نمونه‌ی اجرا با قد/وزن کاربر
user_height_cm = 184
user_weight_kg = 98   # ← این را کاربر وارد کند

# 1) محاسبه‌ی ضریب پهنا از BMI
ws, bmi = width_scale_from_weight_height(user_weight_kg, user_height_cm)

# 2) اجرای pipeline با width_scale خودکار
tp = np.load("src/pre/img_triplet_/tmp_front_smpl.npz", allow_pickle=True)
results = measure_pipeline(tp, user_height_cm=user_height_cm, gender="male", width_scale=ws)

# 3) نمایش نتیجه + چاپ BMI و ضریب
print(f"BMI: {bmi:.1f}  |  width_scale(auto): {ws:.3f}")
print_measurement_results_farsi(results)


Scaled body height (m): 1.8399999141693115
>>> Using CUSTOM joint map (pelvis=9, hips=8/12, knees=3/4, ankles=7/6, neck=22).
BMI: 28.9  |  width_scale(auto): 0.847
📏 نتایج اندازه‌گیری بدن:
----------------------------------------
دور سینه: 70.8 سانتی‌متر
دور کمر: 75.4 سانتی‌متر
دور باسن: 80.6 سانتی‌متر
دور بازو: 41.7 سانتی‌متر
دور ران: 40.5 سانتی‌متر
قد بیرونی شلوار: 86.6 سانتی‌متر
قد آستین: 70.0 سانتی‌متر
قد بالاتنه (بلوز): 93.4 سانتی‌متر
قد لباس بلند: 181.9 سانتی‌متر
قد کاربر: 184.0 سانتی‌متر
ضریب مقیاس مدل: 1.0 
ضریب پهنای بدن: 0.8 
جنسیت: male
----------------------------------------
